<a href="https://colab.research.google.com/github/Lordvaderani/Machine-Learning-Driven-Quantum-Architecture-Search-for-Optimized-VQE-Ansatzes/blob/main/PIRQAS_multimolecule_IEEE_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PIR-QAS: Physics-Informed Residual Quantum Architecture Search

This notebook is a cleaned research prototype of the proposed pipeline:

**MPS/DMRG warm start → adaptive world model → PPO residual search → physics-aware reward → true-VQE verification**

Key fixes relative to the previous implementation:
- the warm start is produced by an actual DMRG/MPS calculation rather than a fixed `TwoLocal` ansatz;
- the RL reward minimizes energy relative to the warm-start baseline instead of using the exact energy as a training target;
- the dream phase uses only the learned surrogate for energy and physics observables;
- the surrogate is updated with newly verified policy rollouts;
- spin is represented with both `S_z` and `S^2`, with `S^2` used for singlet fidelity;
- LiH uses an explicit active-space reduction to 4 qubits;
- H2/LiH evaluation and metric names are unified;
- the number of true VQE evaluations is tracked explicitly.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab setup
# Keep Colab's preinstalled CUDA-enabled PyTorch; reinstalling torch via pip can replace the CUDA build.
!pip install -q qiskit qiskit-nature pyscf gymnasium stable-baselines3 matplotlib ase quimb


In [4]:
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import StatePreparation
from qiskit.primitives import StatevectorEstimator
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.properties.s_operators import s_z_operator, s_plus_operator, s_minus_operator
from ase import Atoms
import quimb.tensor as qtn

warnings.filterwarnings("ignore", category=DeprecationWarning)
SEED = 7

# Automatically use the Colab NVIDIA GPU for PyTorch/DreamQAS components when available.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_AVAILABLE = torch.cuda.is_available()
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if GPU_AVAILABLE:
    torch.cuda.manual_seed_all(SEED)
    print(f"GPU enabled: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("CUDA GPU not available; PyTorch/DreamQAS will use CPU.")

mapper = JordanWignerMapper()
estimator = StatevectorEstimator()
CHEMICAL_ACCURACY = 1.6e-3
print("Imports ready.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


GPU enabled: Tesla T4
CUDA version: 12.8
Imports ready.


### Compute backend
The notebook automatically detects a Colab NVIDIA GPU and uses CUDA for the PyTorch world model and PPO policy. The quantum-chemistry, Qiskit statevector/VQE, and DMRG/MPS parts remain CPU-bound unless their individual backend supports GPU acceleration.

To verify the device, the import cell prints the GPU name and CUDA version when available.


In [5]:
def expectation(qc, observable):
    return float(np.real(estimator.run([(qc, observable)]).result()[0].data.evs))


def make_spin_observables(problem):
    n_spin_orb = problem.num_spin_orbitals
    n_spatial = problem.num_spatial_orbitals
    num_op = FermionicOp({f"+_{i} -_{i}": 1.0 for i in range(n_spin_orb)}, num_spin_orbitals=n_spin_orb)
    n_q = mapper.map(num_op)
    sz_q = mapper.map(s_z_operator(n_spatial))
    sp_q = mapper.map(s_plus_operator(n_spatial))
    sm_q = mapper.map(s_minus_operator(n_spatial))
    s2_q = sz_q.compose(sz_q) + 0.5 * (sp_q.compose(sm_q) + sm_q.compose(sp_q))
    return n_q, sz_q, s2_q


def build_mps_warm_start(qop, bond_dim=4, cutoff=1e-8):
    """DMRG/MPS warm start, independent of the exact ground-state vector."""
    nqubits = qop.num_qubits
    dense_h = np.asarray(qop.to_matrix(), dtype=complex)
    mpo = qtn.MatrixProductOperator.from_dense(dense_h, dims=[2] * nqubits)
    dmrg = qtn.DMRG2(mpo, bond_dims=[2, bond_dim], cutoffs=cutoff)
    dmrg.solve(tol=1e-7, verbosity=0)
    mps_energy = float(np.real(dmrg.energy))
    mps_state = np.asarray(dmrg.state.to_dense()).reshape(-1)
    mps_state /= np.linalg.norm(mps_state)
    qc = QuantumCircuit(nqubits)
    qc.append(StatePreparation(mps_state, normalize=True), range(nqubits))
    decomp = qc.decompose(reps=10)
    return qc, mps_energy, decomp.depth(), decomp.count_ops().get("cx", 0), mps_state


def make_actions(nq):
    return ([('RY', q) for q in range(nq)] + [('RZ', q) for q in range(nq)] +
            [('CX', q, q+1) for q in range(nq-1)] + [('STOP',)])


def build_addition_circuit(warm_start, actions, history):
    qc = warm_start.copy()
    params = []
    for a_idx in history:
        act = actions[a_idx]
        if act[0] == "RY":
            p = Parameter(f"theta_{len(params)}")
            qc.ry(p, act[1]); params.append(p)
        elif act[0] == "RZ":
            p = Parameter(f"phi_{len(params)}")
            qc.rz(p, act[1]); params.append(p)
        elif act[0] == "CX":
            qc.cx(act[1], act[2])
    return qc, params


def true_vqe_evaluate(record, history, opt_maxiter=60):
    actions = make_actions(record["qubit_op"].num_qubits)
    qc, params = build_addition_circuit(record["warm_start"], actions, history)
    if params:
        def energy_fn(x):
            bound = qc.assign_parameters(dict(zip(params, x)))
            return expectation(bound, record["qubit_op"])
        res = minimize(energy_fn, np.zeros(len(params)), method="COBYLA", options={"maxiter": opt_maxiter})
        optimal, energy = res.x, float(res.fun)
    else:
        optimal, energy = np.array([]), expectation(qc, record["qubit_op"])
    bound = qc.assign_parameters(dict(zip(params, optimal))) if params else qc
    n_val = expectation(bound, record["n_op"])
    sz_val = expectation(bound, record["sz_op"])
    s2_val = expectation(bound, record["s2_op"])
    decomp = bound.decompose(reps=10)
    return {"energy": energy, "n_value": n_val, "sz_value": sz_val, "s2_value": s2_val,
            "depth": decomp.depth(), "cx": decomp.count_ops().get("cx", 0),
            "history": tuple(history)}

In [6]:
def make_problem_h2(bond_length):
    problem = PySCFDriver(atom=f"H 0 0 0; H 0 0 {bond_length}", basis="sto3g").run()
    qop = mapper.map(problem.hamiltonian.second_q_op())
    exact = float(np.linalg.eigvalsh(qop.to_matrix())[0].real)
    hf = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
    n_op, sz_op, s2_op = make_spin_observables(problem)
    warm, mps_energy, warm_depth, warm_cx, _ = build_mps_warm_start(qop, bond_dim=4)
    return {"name":"H2", "bond_length":float(bond_length), "qubit_op":qop, "exact_energy":exact,
            "hf":hf, "warm_start":warm, "warm_energy":mps_energy, "warm_depth":warm_depth, "warm_cx":warm_cx,
            "n_target":float(sum(problem.num_particles)), "sz_target":0.0, "s2_target":0.0,
            "n_op":n_op, "sz_op":sz_op, "s2_op":s2_op, "actions":None}

BOND_LENGTHS_H2 = np.linspace(0.5, 2.8, 12)
geometry_cache = [make_problem_h2(r) for r in BOND_LENGTHS_H2]
print(f"Prepared {len(geometry_cache)} H2 geometries.")
print("Example warm energy/depth/CNOTs:", geometry_cache[0]["warm_energy"], geometry_cache[0]["warm_depth"], geometry_cache[0]["warm_cx"])

Prepared 12 H2 geometries.
Example warm energy/depth/CNOTs: -2.113514216310622 23 11


In [7]:
class ResidualQASEnv(gym.Env):
    def __init__(self, geometry_cache, surrogate=None, max_residual_depth=4, depth_penalty=0.002, cx_penalty=0.001, opt_maxiter=60):
        super().__init__()
        self.geometry_cache = geometry_cache
        self.surrogate = surrogate
        self.max_depth = max_residual_depth
        self.depth_penalty = depth_penalty
        self.cx_penalty = cx_penalty
        self.opt_maxiter = opt_maxiter
        self.actions = self._make_actions(geometry_cache[0]["qubit_op"].num_qubits)
        self.n_actions = len(self.actions)
        self.action_space = spaces.Discrete(self.n_actions)
        self.observation_space = spaces.Box(low=-20.0, high=20.0, shape=(self.max_depth*self.n_actions+4,), dtype=np.float32)
        self.history = []

    @staticmethod
    def _make_actions(nq):
        return make_actions(nq)

    def set_geometry(self, record):
        self.record = record
        self.bond_length = record["bond_length"]

    def _obs(self):
        h = np.zeros((self.max_depth, self.n_actions), dtype=np.float32)
        for i, a in enumerate(self.history[:self.max_depth]): h[i, a] = 1.0
        meta = np.array([self.bond_length, self.record["warm_energy"], self.record["warm_depth"], self.record["warm_cx"]], dtype=np.float32)
        return np.concatenate([h.flatten(), meta])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.set_geometry(self.geometry_cache[self.np_random.integers(len(self.geometry_cache))])
        self.history = []
        return self._obs(), {"bond_length":self.bond_length}

    def set_surrogate(self, surrogate): self.surrogate = surrogate

    def _predict(self):
        x = torch.tensor(self._obs(), dtype=torch.float32, device=DEVICE).unsqueeze(0)
        self.surrogate.eval()
        with torch.no_grad(): y = self.surrogate(x).squeeze(0).detach().cpu().numpy()
        return {"energy":float(y[0]), "n_value":float(y[1]), "sz_value":float(y[2]), "s2_value":float(y[3])}

    def _terminal_result(self):
        if self.surrogate is not None:
            out = self._predict()
            out.update({"history":tuple(self.history), "depth":len(self.history)+self.record["warm_depth"],
                        "cx":sum(self.actions[a][0]=='CX' for a in self.history)+self.record["warm_cx"], "true_eval":False})
            return out
        out = true_vqe_evaluate(self.record, self.history, self.opt_maxiter); out["true_eval"] = True; return out

    def step(self, action):
        act = self.actions[int(action)]
        terminated = act[0] == 'STOP'; truncated = False
        if not terminated:
            self.history.append(int(action))
            if len(self.history) >= self.max_depth: terminated, truncated = True, True
        reward, info = 0.0, {}
        if terminated:
            result = self._terminal_result()

            # 1. Calculate raw physics violations
            n_pen = (result['n_value'] - self.record['n_target'])**2
            sz_pen = (result['sz_value'] - self.record['sz_target'])**2
            s2_pen = (result['s2_value'] - self.record['s2_target'])**2

            # 2. Calculate raw energy improvement (E_warm - E_candidate)
            improvement = self.record['warm_energy'] - result['energy']

            # --- THE FIX: REBALANCED REWARD ALGEBRA ---
            # Multiply energy improvement by 50 to make it competitive with integer penalties
            # Reduce physics lambdas so they guide the agent without paralyzing it
            ENERGY_SCALE = 50.0

            reward = ((ENERGY_SCALE * improvement) -
                     (0.05 * n_pen) - (0.05 * sz_pen) - (0.10 * s2_pen) -
                     (self.depth_penalty * result['depth']) -
                     (self.cx_penalty * result['cx']))
            # ------------------------------------------

            info = {
                **result,
                'energy_improvement': improvement,
                'penalty_n': 0.05 * n_pen,
                'penalty_sz': 0.05 * sz_pen,
                'penalty_s2': 0.10 * s2_pen,
                'bond_length': self.bond_length
            }
        # Added return statement to fix TypeError
        return self._obs(), reward, terminated, truncated, info

In [8]:
class MultiOutputWorldModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim,256), nn.ReLU(), nn.Linear(256,128), nn.ReLU(), nn.Linear(128,4))
    def forward(self,x): return self.net(x)


def train_world_model(X,Y,epochs=250,lr=3e-3):
    model = MultiOutputWorldModel(X.shape[1]).to(DEVICE); opt = optim.Adam(model.parameters(), lr=lr)
    X = X.to(DEVICE); Y = Y.to(DEVICE)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = nn.functional.mse_loss(model(X),Y); loss.backward(); opt.step()
    return model, float(loss.detach())


def collect_true_dataset(env,n_episodes=200):
    X,Y=[],[]
    for _ in range(n_episodes):
        obs,_=env.reset(); done=False
        while not done: obs,_,done,_,info=env.step(env.action_space.sample())
        X.append(obs); Y.append([info['energy'],info['n_value'],info['sz_value'],info['s2_value']])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

true_env = ResidualQASEnv(geometry_cache, surrogate=None, max_residual_depth=4)
X_np,Y_np = collect_true_dataset(true_env, n_episodes=200)
X,Y = torch.tensor(X_np, dtype=torch.float32),torch.tensor(Y_np, dtype=torch.float32)
world_model,wm_loss = train_world_model(X,Y)
print(f"Initial world model: MSE={wm_loss:.3e}; real VQE calls={len(X_np)}")

Initial world model: MSE=1.937e-03; real VQE calls=200


In [ ]:
def train_ppo_with_world_model(world_model, geometry_cache, timesteps=6000, seed=SEED):
    dream_env = Monitor(ResidualQASEnv(geometry_cache, surrogate=world_model, max_residual_depth=4))
    ppo_device = 'cuda' if GPU_AVAILABLE else 'cpu'
    model = PPO('MlpPolicy', dream_env, n_steps=256, batch_size=64, ent_coef=0.05, learning_rate=1e-3, gamma=0.99, verbose=0, seed=seed, device=ppo_device)
    print(f'PPO device: {ppo_device}')
    model.learn(total_timesteps=timesteps)
    return model

policy = train_ppo_with_world_model(world_model, geometry_cache)
print('Initial world-model PPO training complete.')

In [ ]:
def evaluate_policy_true(policy, geometry_cache, candidates_per_geometry=3):
    verified=[]; total_calls=0; env=ResidualQASEnv(geometry_cache, surrogate=None, max_residual_depth=4)
    for record in geometry_cache:
        env.set_geometry(record); candidate_histories=[]
        for _ in range(candidates_per_geometry):
            env.history=[]; obs=env._obs(); history=[]
            for _step in range(env.max_depth):
                action,_=policy.predict(obs,deterministic=False); action=int(action); act=env.actions[action]
                if act[0]=='STOP': break
                history.append(action); env.history=history.copy(); obs=env._obs()
            candidate_histories.append(tuple(history))
        results=[]
        for hist in candidate_histories:
            results.append(true_vqe_evaluate(record,hist,opt_maxiter=60)); total_calls += 1
        best=min(results,key=lambda z:z['energy'])
        verified.append({'record':record,'result':best})
    return verified,total_calls

verified_round1,calls_round1 = evaluate_policy_true(policy,geometry_cache,candidates_per_geometry=3)
print(f'Policy verification used {calls_round1} true VQE evaluations.')

In [ ]:
new_X,new_Y=[],[]
for item in verified_round1:
    record,item_result=item['record'],item['result']
    true_env.set_geometry(record); true_env.history=list(item_result['history'])
    new_X.append(true_env._obs()); new_Y.append([item_result['energy'],item_result['n_value'],item_result['sz_value'],item_result['s2_value']])
X_np=np.vstack([X_np,np.asarray(new_X,np.float32)]); Y_np=np.vstack([Y_np,np.asarray(new_Y,np.float32)])
X,Y=torch.tensor(X_np,dtype=torch.float32),torch.tensor(Y_np,dtype=torch.float32)
world_model,wm_loss=train_world_model(X,Y,epochs=300)
policy=train_ppo_with_world_model(world_model,geometry_cache)
print(f'World model updated. Dataset={len(X_np)}; MSE={wm_loss:.3e}.')

In [ ]:
# Final H2 verification and research diagnostics
verified_h2,final_true_calls=evaluate_policy_true(policy,geometry_cache,candidates_per_geometry=5)
rows=[]
for item in verified_h2:
    r,z=item['record'],item['result']
    warm_error=abs(r['warm_energy']-r['exact_energy'])
    final_error=abs(z['energy']-r['exact_energy'])
    residual_depth=z['depth']-r['warm_depth']
    residual_cx=z['cx']-r['warm_cx']
    recovered=(r['warm_energy']-z['energy'])/(r['warm_energy']-r['exact_energy']+1e-12)
    rows.append({
        'bond_length':r['bond_length'], 'exact_energy':r['exact_energy'],
        'warm_energy':r['warm_energy'], 'final_energy':z['energy'],
        'warm_error':warm_error, 'error':final_error,
        'energy_improvement':r['warm_energy']-z['energy'],
        'residual_correlation_recovered':recovered,
        'warm_depth':r['warm_depth'], 'final_depth':z['depth'], 'residual_depth':residual_depth,
        'warm_cx':r['warm_cx'], 'final_cx':z['cx'], 'residual_cx':residual_cx,
        'N_error':abs(z['n_value']-r['n_target']),
        'Sz_error':abs(z['sz_value']-r['sz_target']),
        'S2_error':abs(z['s2_value']-r['s2_target']),
        'chemical_accuracy':final_error <= CHEMICAL_ACCURACY
    })

# Aggregate research metrics
h2_warm_errors=np.array([x['warm_error'] for x in rows])
h2_errors=np.array([x['error'] for x in rows])
h2_improvements=np.array([x['energy_improvement'] for x in rows])
h2_recovery=np.array([x['residual_correlation_recovered'] for x in rows])
h2_warm_depth=np.array([x['warm_depth'] for x in rows],dtype=float)
h2_final_depth=np.array([x['final_depth'] for x in rows],dtype=float)
h2_res_depth=np.array([x['residual_depth'] for x in rows],dtype=float)
h2_warm_cx=np.array([x['warm_cx'] for x in rows],dtype=float)
h2_final_cx=np.array([x['final_cx'] for x in rows],dtype=float)
h2_res_cx=np.array([x['residual_cx'] for x in rows],dtype=float)
h2_nerr=np.array([x['N_error'] for x in rows])
h2_szerr=np.array([x['Sz_error'] for x in rows])
h2_s2err=np.array([x['S2_error'] for x in rows])

print('--- H2 Research Diagnostics ---')
print(f'Warm-start mean energy error: {np.mean(h2_warm_errors):.3e} Ha')
print(f'Final mean energy error:       {np.mean(h2_errors):.3e} Ha')
print(f'Mean energy improvement:       {np.mean(h2_improvements):.3e} Ha')
print(f'Mean residual correlation recovered: {np.mean(h2_recovery):.2%}')
print(f'Chemical-accuracy success rate: {100*np.mean([x["chemical_accuracy"] for x in rows]):.1f}%')
print(f'Median / max final error: {np.median(h2_errors):.3e} / {np.max(h2_errors):.3e} Ha')
print(f'Warm depth mean: {np.mean(h2_warm_depth):.2f}')
print(f'Final depth mean: {np.mean(h2_final_depth):.2f}')
print(f'Residual depth overhead mean: {np.mean(h2_res_depth):.2f}')
print(f'Warm CNOT mean: {np.mean(h2_warm_cx):.2f}')
print(f'Final CNOT mean: {np.mean(h2_final_cx):.2f}')
print(f'Residual CNOT overhead mean: {np.mean(h2_res_cx):.2f}')
print(f'Mean |N-Ntarget|: {np.mean(h2_nerr):.3e}')
print(f'Mean |Sz-Sztarget|: {np.mean(h2_szerr):.3e}')
print(f'Mean |S2-S2target|: {np.mean(h2_s2err):.3e}')
print(f'Final H2 verification calls: {final_true_calls}')
print(f'Total H2 true VQE evaluations incl. dataset+verification: {len(X_np)+calls_round1+final_true_calls}')

bl=np.array([x['bond_length'] for x in rows])
order=np.argsort(bl); bl=bl[order]
err=h2_errors[order]; warmerr=h2_warm_errors[order]
resd=h2_res_depth[order]; resc=h2_res_cx[order]
rec=h2_recovery[order]

plt.figure(figsize=(8,5))
plt.semilogy(bl,warmerr,marker='o',label='MPS warm start')
plt.semilogy(bl,err,marker='s',label='PIR-QAS final')
plt.axhline(CHEMICAL_ACCURACY,linestyle='--',label='Chemical accuracy')
plt.xlabel('H-H bond length (Å)'); plt.ylabel('Energy error vs exact (Ha)')
plt.title('H2: Warm Start vs PIR-QAS Accuracy'); plt.legend(); plt.show()

plt.figure(figsize=(8,5))
plt.plot(bl,resd,marker='o',label='Residual depth added by RL')
plt.plot(bl,resc,marker='s',label='Residual CNOTs added by RL')
plt.xlabel('H-H bond length (Å)'); plt.ylabel('Residual resource count')
plt.title('H2: Residual Circuit Overhead'); plt.legend(); plt.show()

plt.figure(figsize=(8,5))
plt.plot(bl,h2_improvements[order],marker='o',label='Energy improvement')
plt.xlabel('H-H bond length (Å)'); plt.ylabel('Warm energy - final energy (Ha)')
plt.title('H2: Energy Improvement from Residual Search'); plt.legend(); plt.show()

plt.figure(figsize=(8,5))
plt.plot(bl,rec,marker='o',label='Residual correlation recovered')
plt.axhline(1.0,linestyle='--',label='100% recovery')
plt.xlabel('H-H bond length (Å)'); plt.ylabel('Recovered fraction')
plt.title('H2: Residual Correlation Recovery'); plt.legend(); plt.show()

plt.figure(figsize=(8,5))
plt.semilogy(bl,h2_nerr[order],marker='o',label='|N-Ntarget|')
plt.semilogy(bl,h2_szerr[order]+1e-16,marker='s',label='|Sz-Sztarget|')
plt.semilogy(bl,h2_s2err[order]+1e-16,marker='^',label='|S2-S2target|')
plt.xlabel('H-H bond length (Å)'); plt.ylabel('Physics violation')
plt.title('H2: Physics Constraint Diagnostics'); plt.legend(); plt.show()


## LiH: active-space scaling

The previous notebook mixed a 4-qubit symmetry operator with the full STO-3G LiH Hamiltonian. This version applies an explicit active-space reduction first so the Hamiltonian and all observables have matching register sizes.

The default active space keeps 2 electrons in 2 spatial orbitals (4 spin orbitals).

In [13]:
def make_problem_lih(bond_length):
    atoms=Atoms('LiH',positions=[[0,0,0],[0,0,bond_length]])
    atom_str='; '.join(f'{s} {p[0]} {p[1]} {p[2]}' for s,p in zip(atoms.get_chemical_symbols(),atoms.get_positions()))
    full_problem=PySCFDriver(atom=atom_str,basis='sto3g').run()
    problem=ActiveSpaceTransformer(2,2).transform(full_problem)
    qop=mapper.map(problem.hamiltonian.second_q_op())
    exact=float(np.linalg.eigvalsh(qop.to_matrix())[0].real)
    hf=HartreeFock(problem.num_spatial_orbitals,problem.num_particles,mapper)
    n_op,sz_op,s2_op=make_spin_observables(problem)
    warm,mps_energy,warm_depth,warm_cx,_=build_mps_warm_start(qop,bond_dim=4)
    return {'name':'LiH','bond_length':float(bond_length),'qubit_op':qop,'exact_energy':exact,'hf':hf,'warm_start':warm,
            'warm_energy':mps_energy,'warm_depth':warm_depth,'warm_cx':warm_cx,'n_target':float(sum(problem.num_particles)),
            'sz_target':0.0,'s2_target':0.0,'n_op':n_op,'sz_op':sz_op,'s2_op':s2_op,'actions':None}

BOND_LENGTHS_LiH=np.linspace(1.0,3.0,8)
geometry_cache_LiH=[make_problem_lih(r) for r in BOND_LENGTHS_LiH]
print(f'Prepared {len(geometry_cache_LiH)} LiH active-space geometries.')
print('LiH qubits:',geometry_cache_LiH[0]['qubit_op'].num_qubits)

Prepared 8 LiH active-space geometries.
LiH qubits: 4


In [ ]:
true_env_LiH=ResidualQASEnv(geometry_cache_LiH,surrogate=None,max_residual_depth=4)
X_lih,Y_lih=collect_true_dataset(true_env_LiH,n_episodes=150)
X_lih_t,Y_lih_t=torch.tensor(X_lih,dtype=torch.float32),torch.tensor(Y_lih,dtype=torch.float32)
world_model_LiH,wm_loss_LiH=train_world_model(X_lih_t,Y_lih_t)
policy_LiH=train_ppo_with_world_model(world_model_LiH,geometry_cache_LiH,timesteps=5000)
verified_lih,calls_lih=evaluate_policy_true(policy_LiH,geometry_cache_LiH,candidates_per_geometry=3)

lih_rows=[]
for item in verified_lih:
    r,z=item['record'],item['result']
    final_error=abs(z['energy']-r['exact_energy'])
    warm_error=abs(r['warm_energy']-r['exact_energy'])
    lih_rows.append({
        'bond_length':r['bond_length'], 'warm_error':warm_error, 'final_error':final_error,
        'improvement':r['warm_energy']-z['energy'],
        'warm_depth':r['warm_depth'], 'final_depth':z['depth'], 'residual_depth':z['depth']-r['warm_depth'],
        'warm_cx':r['warm_cx'], 'final_cx':z['cx'], 'residual_cx':z['cx']-r['warm_cx'],
        'N_error':abs(z['n_value']-r['n_target']), 'Sz_error':abs(z['sz_value']-r['sz_target']),
        'S2_error':abs(z['s2_value']-r['s2_target'])
    })

lih_err=np.array([x['final_error'] for x in lih_rows]); lih_warm_err=np.array([x['warm_error'] for x in lih_rows])
lih_s2=np.array([x['S2_error'] for x in lih_rows]); lih_n=np.array([x['N_error'] for x in lih_rows]); lih_sz=np.array([x['Sz_error'] for x in lih_rows])
lih_resd=np.array([x['residual_depth'] for x in lih_rows]); lih_resc=np.array([x['residual_cx'] for x in lih_rows])

print('--- LiH Active-Space Diagnostics ---')
print(f'LiH mean warm-start error: {np.mean(lih_warm_err):.3e} Ha')
print(f'LiH mean final error:      {np.mean(lih_err):.3e} Ha')
print(f'LiH median / max error:    {np.median(lih_err):.3e} / {np.max(lih_err):.3e} Ha')
print(f'LiH chemical-accuracy success rate: {100*np.mean(lih_err <= CHEMICAL_ACCURACY):.1f}%')
print(f'LiH mean residual depth: {np.mean(lih_resd):.2f}')
print(f'LiH mean residual CNOTs: {np.mean(lih_resc):.2f}')
print(f'LiH mean |N-Ntarget|: {np.mean(lih_n):.3e}')
print(f'LiH mean |Sz-Sztarget|: {np.mean(lih_sz):.3e}')
print(f'LiH mean |S2-S2target|: {np.mean(lih_s2):.3e}')
print(f'LiH final verification calls: {calls_lih}')
print(f'LiH total true VQE evaluations incl. dataset: {len(X_lih)+calls_lih}')

bl_lih=np.array([x['bond_length'] for x in lih_rows]); order_lih=np.argsort(bl_lih); bl_lih=bl_lih[order_lih]
plt.figure(figsize=(8,5)); plt.semilogy(bl_lih,lih_warm_err[order_lih],marker='o',label='MPS warm start'); plt.semilogy(bl_lih,lih_err[order_lih],marker='s',label='PIR-QAS final'); plt.axhline(CHEMICAL_ACCURACY,linestyle='--',label='Chemical accuracy'); plt.xlabel('Li-H bond length (Å)'); plt.ylabel('Energy error vs exact (Ha)'); plt.title('LiH Active Space: Warm Start vs PIR-QAS'); plt.legend(); plt.show()
plt.figure(figsize=(8,5)); plt.plot(bl_lih,lih_resd[order_lih],marker='o',label='Residual depth'); plt.plot(bl_lih,lih_resc[order_lih],marker='s',label='Residual CNOTs'); plt.xlabel('Li-H bond length (Å)'); plt.ylabel('Residual resource count'); plt.title('LiH Active Space: Residual Circuit Overhead'); plt.legend(); plt.show()


In [ ]:
print('\n=== PIR-QAS SUMMARY ===')
print('PyTorch device:', DEVICE)
if GPU_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)

print('\n--- H2 accuracy ---')
print('Warm-start mean error:',f"{np.mean(h2_warm_errors):.3e} Ha")
print('Final mean error:',f"{np.mean(h2_errors):.3e} Ha")
print('Median / max final error:',f"{np.median(h2_errors):.3e} / {np.max(h2_errors):.3e} Ha")
print('Mean energy improvement:',f"{np.mean(h2_improvements):.3e} Ha")
print('Residual correlation recovered:',f"{np.mean(h2_recovery):.2%}")
print('Chemical-accuracy success rate:',f"{100*np.mean(h2_errors <= CHEMICAL_ACCURACY):.1f}%")

print('\n--- H2 resources ---')
print('Mean warm depth:',f"{np.mean(h2_warm_depth):.2f}")
print('Mean final depth:',f"{np.mean(h2_final_depth):.2f}")
print('Mean residual depth:',f"{np.mean(h2_res_depth):.2f}")
print('Mean warm CNOTs:',f"{np.mean(h2_warm_cx):.2f}")
print('Mean final CNOTs:',f"{np.mean(h2_final_cx):.2f}")
print('Mean residual CNOTs:',f"{np.mean(h2_res_cx):.2f}")

print('\n--- H2 physics ---')
print('Mean |N-Ntarget|:',f"{np.mean(h2_nerr):.3e}")
print('Mean |Sz-Sztarget|:',f"{np.mean(h2_szerr):.3e}")
print('Mean |S2-S2target|:',f"{np.mean(h2_s2err):.3e}")

print('\n--- H2 verification cost ---')
print('Initial real-data calls:',len(X_np))
print('Round-1 verification calls:',calls_round1)
print('Final verification calls:',final_true_calls)
print('Total H2 true VQE evaluations:',len(X_np)+calls_round1+final_true_calls)

print('\n--- LiH active space ---')
print('Mean warm-start error:',f"{np.mean(lih_warm_err):.3e} Ha")
print('Mean final error:',f"{np.mean(lih_err):.3e} Ha")
print('Median / max final error:',f"{np.median(lih_err):.3e} / {np.max(lih_err):.3e} Ha")
print('Chemical-accuracy success rate:',f"{100*np.mean(lih_err <= CHEMICAL_ACCURACY):.1f}%")
print('Mean residual depth:',f"{np.mean(lih_resd):.2f}")
print('Mean residual CNOTs:',f"{np.mean(lih_resc):.2f}")
print('Mean |N-Ntarget|:',f"{np.mean(lih_n):.3e}")
print('Mean |Sz-Sztarget|:',f"{np.mean(lih_sz):.3e}")
print('Mean |S2-S2target|:',f"{np.mean(lih_s2):.3e}")
print('LiH verification calls:',calls_lih)
print('Total LiH true VQE evaluations:',len(X_lih)+calls_lih)

print('\nNote: total depth/CNOT counts include the MPS warm-start circuit; residual depth/CNOTs isolate RL-added overhead.')

# Multi-molecule research benchmark and ablation study

This extension turns the original PIR-QAS prototype into a **B.Tech research benchmark notebook**. It keeps the original H₂/LiH implementation and adds a reproducible framework for comparing molecular systems, recording failure modes, and studying which combinations of components work best.

### Research question
> **How reliably can reinforcement learning discover compact VQE ansätze across different molecular systems, and what factors contribute to successful or failed ansatz searches?**

The benchmark is designed around:
- H₂
- LiH (2e,2o active space)
- BeH₂ (4e,3o active space by default)
- optional H₄ and H₃⁺ extensions
- MPS/DMRG warm start
- learned world model
- PPO residual architecture search
- physics-aware reward
- true-VQE verification
- multiple seeds
- component/permutation ablations
- accuracy, compactness, physics fidelity, verification cost, and failure diagnostics

**Important:** DreamQAS and TensorRL-QAS are included as literature baselines/references here, not claimed as implementations of this notebook. TensorRL-QAS explicitly combines tensor-network warm starts with RL, while DreamQAS uses a learned decision-useful world model and selective real-VQE verification. The present notebook should therefore be described as a related prototype/benchmark, not as an independent reimplementation of those papers.


In [ ]:
# =========================
# RESEARCH BENCHMARK CONFIG
# =========================
# Set QUICK_MODE=True for a practical Colab smoke test.
# Set QUICK_MODE=False for the larger paper-style run.
QUICK_MODE = True

if QUICK_MODE:
    DATASET_EPISODES = 40
    WORLD_EPOCHS = 120
    PPO_STEPS = 1500
    CANDIDATES_PER_GEOMETRY = 2
    SEEDS = [7]
else:
    DATASET_EPISODES = 200
    WORLD_EPOCHS = 300
    PPO_STEPS = 6000
    CANDIDATES_PER_GEOMETRY = 5
    SEEDS = [7, 17, 27]

# Molecule coverage. H4/H3+ are optional because their chemistry setup is
# more expensive and can be enabled after the core H2/LiH/BeH2 benchmark.
RUN_H2 = True
RUN_LIH = True
RUN_BEH2 = True
RUN_H4 = True
RUN_H3PLUS = True

print('Benchmark configuration:')
print({
    'quick_mode': QUICK_MODE,
    'dataset_episodes': DATASET_EPISODES,
    'world_epochs': WORLD_EPOCHS,
    'ppo_steps': PPO_STEPS,
    'candidates_per_geometry': CANDIDATES_PER_GEOMETRY,
    'seeds': SEEDS
})


In [ ]:
# =========================
# GENERIC MOLECULAR PROBLEM BUILDER
# =========================

def make_problem_generic(name, atom_string, bond_parameter=None, charge=0, spin=0,
                         active_electrons=None, active_orbitals=None, bond_dim=4):
    """Build a Qiskit Nature Hamiltonian + MPS warm start for a molecule.

    The optional active-space transformation is applied before mapping, so
    Hamiltonian, number, Sz and S^2 operators all act on the same qubit register.
    """
    full_problem = PySCFDriver(
        atom=atom_string, basis='sto3g', charge=charge, spin=spin
    ).run()

    problem = full_problem
    if active_electrons is not None and active_orbitals is not None:
        problem = ActiveSpaceTransformer(active_electrons, active_orbitals).transform(full_problem)

    qop = mapper.map(problem.hamiltonian.second_q_op())
    exact = float(np.linalg.eigvalsh(qop.to_matrix())[0].real)
    hf = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
    n_op, sz_op, s2_op = make_spin_observables(problem)
    warm, mps_energy, warm_depth, warm_cx, _ = build_mps_warm_start(qop, bond_dim=bond_dim)

    return {
        'name': name,
        'bond_length': float(bond_parameter if bond_parameter is not None else 0.0),
        'geometry_label': atom_string,
        'qubit_op': qop,
        'exact_energy': exact,
        'hf': hf,
        'warm_start': warm,
        'warm_energy': mps_energy,
        'warm_depth': warm_depth,
        'warm_cx': warm_cx,
        'n_target': float(sum(problem.num_particles)),
        'sz_target': 0.0,
        's2_target': 0.0,
        'n_op': n_op,
        'sz_op': sz_op,
        's2_op': s2_op,
        'actions': None
    }


def make_h2_cache(lengths=None):
    lengths = np.linspace(0.5, 2.8, 12) if lengths is None else lengths
    return [make_problem_generic('H2', f'H 0 0 0; H 0 0 {r}', r, active_electrons=None, active_orbitals=None) for r in lengths]

def make_lih_cache(lengths=None):
    lengths = np.linspace(1.0, 3.0, 8) if lengths is None else lengths
    out=[]
    for r in lengths:
        atoms=Atoms('LiH', positions=[[0,0,0],[0,0,r]])
        atom_str='; '.join(f'{s} {p[0]} {p[1]} {p[2]}' for s,p in zip(atoms.get_chemical_symbols(),atoms.get_positions()))

        # --- THE FIX: EXPAND TO 6 QUBITS ---
        # Arguments: ActiveSpaceTransformer(num_electrons, num_spatial_orbitals)
        # 2 active electrons across 3 spatial orbitals = 6 spin-orbitals (6 qubits)
        out.append(make_problem_generic('LiH', atom_str, r, active_electrons=2, active_orbitals=3))
        # -----------------------------------
    return out

def make_beh2_cache(lengths=None):
    lengths = np.linspace(1.1, 2.0, 6) if lengths is None else lengths
    out=[]
    # Linear H-Be-H geometry. 4e,3o active space -> 6 qubits.
    for r in lengths:
        atom_str=f'Be 0 0 0; H 0 0 {r}; H 0 0 {-r}'
        out.append(make_problem_generic('BeH2', atom_str, r, active_electrons=4, active_orbitals=3))
    return out

def make_h4_cache(lengths=None):
    lengths = np.linspace(0.9, 1.8, 5) if lengths is None else lengths
    out=[]
    # Linear H4 chain. 4e,4o active space -> 8 qubits.
    for r in lengths:
        atom_str='; '.join([f'H 0 0 {k*r}' for k in range(4)])
        out.append(make_problem_generic('H4', atom_str, r, active_electrons=4, active_orbitals=4))
    return out

def make_h3plus_cache(lengths=None):
    lengths = np.linspace(0.8, 1.8, 5) if lengths is None else lengths
    out=[]
    # Simple linear H3+ demonstration. Charge +1, singlet. 2e active space.
    for r in lengths:
        atom_str=f'H 0 0 {-r}; H 0 0 0; H 0 0 {r}'
        out.append(make_problem_generic('H3+', atom_str, r, charge=1, spin=0, active_electrons=2, active_orbitals=2))
    return out

print('Generic molecule builders defined.')

In [ ]:
# =========================
# STANDARDIZED EXPERIMENT RUNNER
# =========================

def run_pirqas_experiment(name, geometry_cache, seed=7, dataset_episodes=None,
                          world_epochs=None, ppo_steps=None, candidates_per_geometry=None):
    """Run the same pipeline for one molecular system and return row-level metrics."""
    dataset_episodes = DATASET_EPISODES if dataset_episodes is None else dataset_episodes
    world_epochs = WORLD_EPOCHS if world_epochs is None else world_epochs
    ppo_steps = PPO_STEPS if ppo_steps is None else ppo_steps
    candidates_per_geometry = CANDIDATES_PER_GEOMETRY if candidates_per_geometry is None else candidates_per_geometry

    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    if GPU_AVAILABLE: torch.cuda.manual_seed_all(seed)

    env = ResidualQASEnv(geometry_cache, surrogate=None, max_residual_depth=4)
    X_np, Y_np = collect_true_dataset(env, n_episodes=dataset_episodes)
    X_t, Y_t = torch.tensor(X_np,dtype=torch.float32), torch.tensor(Y_np,dtype=torch.float32)
    wm, wm_loss = train_world_model(X_t,Y_t,epochs=world_epochs)
    policy = train_ppo_with_world_model(wm,geometry_cache,timesteps=ppo_steps,seed=seed)

    verified, verification_calls = evaluate_policy_true(policy,geometry_cache,candidates_per_geometry=candidates_per_geometry)

    rows=[]
    for item in verified:
        r,z=item['record'],item['result']
        warm_error=abs(r['warm_energy']-r['exact_energy'])
        final_error=abs(z['energy']-r['exact_energy'])
        residual_depth=z['depth']-r['warm_depth']
        residual_cx=z['cx']-r['warm_cx']
        denominator=(r['warm_energy']-r['exact_energy'])
        recovered=(r['warm_energy']-z['energy'])/(denominator+1e-12)
        rows.append({
            'molecule':name,'seed':seed,'bond_parameter':r['bond_length'],
            'qubits':r['qubit_op'].num_qubits,'exact_energy':r['exact_energy'],
            'warm_energy':r['warm_energy'],'final_energy':z['energy'],
            'warm_error':warm_error,'final_error':final_error,
            'energy_improvement':r['warm_energy']-z['energy'],
            'recovered_fraction':recovered,
            'warm_depth':r['warm_depth'],'final_depth':z['depth'],'residual_depth':residual_depth,
            'warm_cx':r['warm_cx'],'final_cx':z['cx'],'residual_cx':residual_cx,
            'N_error':abs(z['n_value']-r['n_target']),
            'Sz_error':abs(z['sz_value']-r['sz_target']),
            'S2_error':abs(z['s2_value']-r['s2_target']),
            'chemical_accuracy':bool(final_error <= CHEMICAL_ACCURACY),
            'verification_calls':verification_calls,
            'dataset_calls':len(X_np),
            'world_model_mse':wm_loss,
            'history_length':len(z['history']),
            'history':str(z['history'])
        })
    return rows, {'world_model_mse':wm_loss,'dataset_calls':len(X_np),'verification_calls':verification_calls,'policy':policy}

# =========================
# GENERIC MOLECULAR PROBLEM BUILDER (Moved from jNz8FBpyA-40)
# =========================

def make_problem_generic(name, atom_string, bond_parameter=None, charge=0, spin=0,
                         active_electrons=None, active_orbitals=None, bond_dim=4):
    """Build a Qiskit Nature Hamiltonian + MPS warm start for a molecule.

    The optional active-space transformation is applied before mapping, so
    Hamiltonian, number, Sz and S^2 operators all act on the same qubit register.
    """
    full_problem = PySCFDriver(
        atom=atom_string, basis='sto3g', charge=charge, spin=spin
    ).run()

    problem = full_problem
    if active_electrons is not None and active_orbitals is not None:
        problem = ActiveSpaceTransformer(active_electrons, active_orbitals).transform(full_problem)

    qop = mapper.map(problem.hamiltonian.second_q_op())
    exact = float(np.linalg.eigvalsh(qop.to_matrix())[0].real)
    hf = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
    n_op, sz_op, s2_op = make_spin_observables(problem)
    warm, mps_energy, warm_depth, warm_cx, _ = build_mps_warm_start(qop, bond_dim=bond_dim)

    return {
        'name': name,
        'bond_length': float(bond_parameter if bond_parameter is not None else 0.0),
        'geometry_label': atom_string,
        'qubit_op': qop,
        'exact_energy': exact,
        'hf': hf,
        'warm_start': warm,
        'warm_energy': mps_energy,
        'warm_depth': warm_depth,
        'warm_cx': warm_cx,
        'n_target': float(sum(problem.num_particles)),
        'sz_target': 0.0,
        's2_target': 0.0,
        'n_op': n_op,
        'sz_op': sz_op,
        's2_op': s2_op,
        'actions': None
    }


def make_h2_cache(lengths=None):
    lengths = np.linspace(0.5, 2.8, 12) if lengths is None else lengths
    return [make_problem_generic('H2', f'H 0 0 0; H 0 0 {r}', r, active_electrons=None, active_orbitals=None) for r in lengths]

def make_lih_cache(lengths=None):
    lengths = np.linspace(1.0, 3.0, 8) if lengths is None else lengths
    out=[]
    for r in lengths:
        atoms=Atoms('LiH', positions=[[0,0,0],[0,0,r]])
        atom_str='; '.join(f'{s} {p[0]} {p[1]} {p[2]}' for s,p in zip(atoms.get_chemical_symbols(),atoms.get_positions()))

        # --- THE FIX: EXPAND TO 6 QUBITS ---
        # Arguments: ActiveSpaceTransformer(num_electrons, num_spatial_orbitals)
        # 2 active electrons across 3 spatial orbitals = 6 spin-orbitals (6 qubits)
        out.append(make_problem_generic('LiH', atom_str, r, active_electrons=2, active_orbitals=3))
        # -----------------------------------
    return out

def make_beh2_cache(lengths=None):
    lengths = np.linspace(1.1, 2.0, 6) if lengths is None else lengths
    out=[]
    # Linear H-Be-H geometry. 4e,3o active space -> 6 qubits.
    for r in lengths:
        atom_str=f'Be 0 0 0; H 0 0 {r}; H 0 0 {-r}'
        out.append(make_problem_generic('BeH2', atom_str, r, active_electrons=4, active_orbitals=3))
    return out

def make_h4_cache(lengths=None):
    lengths = np.linspace(0.9, 1.8, 5) if lengths is None else lengths
    out=[]
    # Linear H4 chain. 4e,4o active space -> 8 qubits.
    for r in lengths:
        atom_str='; '.join([f'H 0 0 {k*r}' for k in range(4)])
        out.append(make_problem_generic('H4', atom_str, r, active_electrons=4, active_orbitals=4))
    return out

def make_h3plus_cache(lengths=None):
    lengths = np.linspace(0.8, 1.8, 5) if lengths is None else lengths
    out=[]
    # Simple linear H3+ demonstration. Charge +1, singlet. 2e active space.
    for r in lengths:
        atom_str=f'H 0 0 {-r}; H 0 0 0; H 0 0 {r}'
        out.append(make_problem_generic('H3+', atom_str, r, charge=1, spin=0, active_electrons=2, active_orbitals=2))
    return out


# Build requested systems.
benchmarks={}

# Redefine RUN_* flags to ensure they are available in this cell
# This was removed in a previous edit, but re-adding for robustness.
RUN_H2 = True
RUN_LIH = True
RUN_BEH2 = True
RUN_H4 = True
RUN_H3PLUS = True

if RUN_H2:
    benchmarks['H2']=make_h2_cache()
if RUN_LIH:
    benchmarks['LiH']=make_lih_cache()
if RUN_BEH2:
    benchmarks['BeH2']=make_beh2_cache()
if RUN_H4:
    benchmarks['H4']=make_h4_cache()
if RUN_H3PLUS:
    benchmarks['H3+']=make_h3plus_cache()

print('Systems:', {k: (len(v), v[0]['qubit_op'].num_qubits) for k,v in benchmarks.items()})


In [ ]:
# =========================
# RUN MULTI-SEED BENCHMARK
# =========================
all_rows=[]
run_metadata=[]

for molecule, cache in benchmarks.items():
    print(f'\n===== {molecule} =====')
    for seed in SEEDS:
        print(f'-- seed {seed} --')
        try:
            rows_m, meta_m = run_pirqas_experiment(molecule, cache, seed=seed)
            all_rows.extend(rows_m)
            run_metadata.append({'molecule':molecule,'seed':seed,**{k:v for k,v in meta_m.items() if k != 'policy'}})
            print('completed:', len(rows_m), 'geometries')
        except Exception as exc:
            print(f'FAILED {molecule} seed {seed}: {type(exc).__name__}: {exc}')

import pandas as pd
results_df = pd.DataFrame(all_rows)
metadata_df = pd.DataFrame(run_metadata)
print('\nTotal result rows:', len(results_df))
results_df.head()


In [ ]:
# =========================
# RESEARCH METRICS
# =========================

def summarize_results(df):
    if df.empty: return pd.DataFrame()
    g=df.groupby('molecule')
    summary=g.agg(
        qubits=('qubits','first'),
        geometries=('bond_parameter','count'),
        mean_warm_error=('warm_error','mean'),
        mean_final_error=('final_error','mean'),
        median_final_error=('final_error','median'),
        max_final_error=('final_error','max'),
        chemical_accuracy_rate=('chemical_accuracy','mean'),
        mean_energy_improvement=('energy_improvement','mean'),
        mean_recovered_fraction=('recovered_fraction','mean'),
        mean_warm_depth=('warm_depth','mean'),
        mean_final_depth=('final_depth','mean'),
        mean_residual_depth=('residual_depth','mean'),
        mean_warm_cx=('warm_cx','mean'),
        mean_final_cx=('final_cx','mean'),
        mean_residual_cx=('residual_cx','mean'),
        mean_N_error=('N_error','mean'),
        mean_Sz_error=('Sz_error','mean'),
        mean_S2_error=('S2_error','mean'),
        mean_dataset_calls=('dataset_calls','mean'),
        mean_verification_calls=('verification_calls','mean')
    ).reset_index()
    summary['chemical_accuracy_rate']*=100
    return summary

summary_df=summarize_results(results_df)
summary_df


In [ ]:
# =========================
# FAILURE / SUCCESS DIAGNOSTICS
# =========================

if not results_df.empty:
    results_df['failure_flag'] = ~results_df['chemical_accuracy']
    results_df['no_improvement'] = results_df['energy_improvement'] <= 1e-8
    results_df['physics_failure'] = (results_df['Sz_error'] > 1e-2) | (results_df['S2_error'] > 1e-2)
    results_df['resource_heavy'] = results_df['residual_cx'] >= results_df['residual_cx'].median()

    print('Failure analysis:')
    print('Chemical-accuracy failures:', int(results_df['failure_flag'].sum()))
    print('No/negative energy-improvement cases:', int(results_df['no_improvement'].sum()))
    print('Physics-constraint failures:', int(results_df['physics_failure'].sum()))
    print('Worst final-error rows:')
    display(results_df.sort_values('final_error', ascending=False)[[
        'molecule','seed','bond_parameter','qubits','warm_error','final_error',
        'energy_improvement','residual_depth','residual_cx','Sz_error','S2_error','history'
    ]].head(15))


In [ ]:
# =========================
# PLOTS FOR THE PAPER
# =========================

if not results_df.empty:
    for molecule in results_df['molecule'].unique():
        d=results_df[results_df['molecule']==molecule].sort_values('bond_parameter')
        x=d['bond_parameter'].to_numpy()

        plt.figure(figsize=(8,5))
        plt.semilogy(x,d['warm_error'],marker='o',label='MPS warm start')
        plt.semilogy(x,d['final_error'],marker='s',label='PIR-QAS final')
        plt.axhline(CHEMICAL_ACCURACY,linestyle='--',label='Chemical accuracy')
        plt.xlabel('Geometry parameter (Å)'); plt.ylabel('Energy error (Ha)')
        plt.title(f'{molecule}: Accuracy across geometries'); plt.legend(); plt.show()

        plt.figure(figsize=(8,5))
        plt.plot(x,d['residual_depth'],marker='o',label='Residual depth')
        plt.plot(x,d['residual_cx'],marker='s',label='Residual CNOTs')
        plt.xlabel('Geometry parameter (Å)'); plt.ylabel('RL-added resources')
        plt.title(f'{molecule}: Compactness of discovered residual ansatz'); plt.legend(); plt.show()

        plt.figure(figsize=(8,5))
        plt.plot(x,d['energy_improvement'],marker='o',label='Energy improvement')
        plt.axhline(0,linestyle='--')
        plt.xlabel('Geometry parameter (Å)'); plt.ylabel('Warm energy - final energy (Ha)')
        plt.title(f'{molecule}: Search success / failure'); plt.legend(); plt.show()

        plt.figure(figsize=(8,5))
        plt.semilogy(x,d['Sz_error']+1e-16,marker='o',label='|Sz-Sz,target|')
        plt.semilogy(x,d['S2_error']+1e-16,marker='s',label='|S2-S2,target|')
        plt.xlabel('Geometry parameter (Å)'); plt.ylabel('Physics violation')
        plt.title(f'{molecule}: Symmetry diagnostics'); plt.legend(); plt.show()

    # Cross-molecule resource/accuracy plot
    plt.figure(figsize=(8,5))
    for molecule,d in results_df.groupby('molecule'):
        plt.scatter(d['mean_final_error'] if 'mean_final_error' in d else d['final_error'], d['final_cx'], label=molecule)
    plt.xlabel('Final energy error (Ha)'); plt.ylabel('Final CNOT count')
    plt.title('Accuracy-resource trade-off across molecular systems'); plt.legend(); plt.show()


## Component-permutation / ablation design

The paper should explicitly test whether success comes from the RL algorithm itself or from the surrounding components. The recommended permutation matrix is:

| ID | MPS warm start | World model | Physics reward | PPO residual search | Purpose |
|---|---|---|---|---|---|
| A | ✓ | ✓ | ✓ | ✓ | Full PIR-QAS |
| B | ✗ | ✓ | ✓ | ✓ | Value of MPS warm start |
| C | ✓ | ✗ | ✓ | ✓ | Value of learned surrogate |
| D | ✓ | ✓ | ✗ | ✓ | Value of physics-informed reward |
| E | ✓ | ✓ | ✓ | ✗ | Non-RL / search-control baseline |

For the B.Tech paper, **A–D are enough**. E can be added later because a fair non-RL search baseline needs a carefully matched evaluation budget.

Do not call every permutation a new algorithm. Treat them as **controlled ablations** of the same pipeline. The primary question is whether each component improves reliability, accuracy, compactness, or evaluation efficiency.


In [ ]:
# =========================
# LIGHTWEIGHT PERMUTATION TABLE
# =========================
# This cell creates the experiment registry. It does not silently claim that
# an ablation has been executed. Change RUN_ABLATIONS=True after the core
# benchmark is stable, then implement/execute the corresponding variants.

ABLATION_REGISTRY = pd.DataFrame([
    {'ID':'A','MPS_warm_start':True,'world_model':True,'physics_reward':True,'PPO':True,'status':'core/full'},
    {'ID':'B','MPS_warm_start':False,'world_model':True,'physics_reward':True,'PPO':True,'status':'planned'},
    {'ID':'C','MPS_warm_start':True,'world_model':False,'physics_reward':True,'PPO':True,'status':'planned'},
    {'ID':'D','MPS_warm_start':True,'world_model':True,'physics_reward':False,'PPO':True,'status':'planned'},
    {'ID':'E','MPS_warm_start':True,'world_model':True,'physics_reward':True,'PPO':False,'status':'future/non-RL baseline'},
])
ABLATION_REGISTRY


## Recommended reporting rule

For every permutation and molecule, report the same metrics:

1. mean/median/max energy error
2. chemical-accuracy success rate
3. mean energy improvement over warm start
4. residual depth and residual CNOTs
5. total depth and total CNOTs
6. \(|N-N_t|\), \(|S_z-S_{z,t}|\), and \(|S^2-S^2_t|\)
7. true VQE evaluations
8. world-model MSE (if applicable)
9. success rate across random seeds
10. number of failed geometries and the corresponding circuit histories

This makes the central paper claim much stronger than reporting only one final energy number. It lets you distinguish **accurate but expensive**, **compact but inaccurate**, and **physically invalid** ansatz searches.


In [ ]:
# =========================
# EXPORT RESULTS
# =========================
# These CSV files can be directly used to build IEEE tables/figures.
if not results_df.empty:
    results_df.to_csv('pirqas_multimolecule_results.csv', index=False)
    summary_df.to_csv('pirqas_multimolecule_summary.csv', index=False)
    ABLATION_REGISTRY.to_csv('pirqas_ablation_registry.csv', index=False)
    print('Saved: pirqas_multimolecule_results.csv')
    print('Saved: pirqas_multimolecule_summary.csv')
    print('Saved: pirqas_ablation_registry.csv')


## References for positioning the work

[1] E.-J. Kuo, Y.-L. L. Fang, and S.-Y. C. Chen, “Quantum Architecture Search via Deep Reinforcement Learning,” 2021. This establishes DRL-based QAS using A2C/PPO for circuit construction.

[2] E. Ye and S.-Y. C. Chen, “Quantum Architecture Search via Continual Reinforcement Learning,” 2021. This extends RL-QAS toward changing noise environments.

[3] A. Kundu and S. Mangini, “TensorRL-QAS: Reinforcement learning with tensor networks for improved quantum architecture search,” NeurIPS 2025. This is particularly relevant because it uses an MPS/tensor-network warm start before RL architecture search.

[4] J. Niu et al., “DreamQAS: Learning a Decision-Useful World Model for VQE-Efficient Quantum Architecture Search,” 2026. This is relevant to the world-model/selective-verification direction.

The notebook does **not** claim to reproduce the numerical results of [3] or [4]. Those works should be treated as related literature and, if implemented later, as explicit baselines with their own code/configuration.
